# Confusion grids for the main model (Multi-Scale PatchTST) on Colab GPU

**Before running:** Runtime → Change runtime type → **L4 GPU**. Then Runtime → **Run all** and keep the tab open (about 15 minutes).

This reruns the three models from the first Colab run, but this time every validation prediction is saved, so we can
compare predictions with what really happened (confusion grids). Results go to `MyDrive/Senior/results/`:
`*_r2.json` (scores + grids), `*_r2_preds.csv.gz` (every prediction) and `results_summary.xlsx` (one sheet per view).
Validation block only; the test block stays sealed.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib
for f in ['scripts/baselines_deep.py', 'scripts/summarize_results.py']:
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
assert 'confusion' in open('scripts/baselines_deep.py', encoding='utf-8').read(), 'stop: push the latest commit from your PC (baselines_deep.py with confusion grids), then run again'

In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
# Files must match the PC copies (hashes recorded 2026-10-15; sentiment file as stored in git, LF line endings)
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
!python tests/test_pipeline.py

In [ ]:
# 1) Main model WITH news (v1) -- the model the confusion grid is for
!python scripts/baselines_deep.py --models multiscale_sector --news data/derived/sentiment_scores.csv --out "$OUT_DIR/main_model_v1_news_r2.json"

In [ ]:
# 2) Main model WITHOUT news (v0), same GPU, for comparison
!python scripts/baselines_deep.py --models multiscale_sector --out "$OUT_DIR/main_model_v0_r2.json"

In [ ]:
# 3) One-scale partner with news (what the multi-scale part adds)
!python scripts/baselines_deep.py --models singlescale_sector --news data/derived/sentiment_scores.csv --out "$OUT_DIR/singlescale_v1_news_r2.json"

In [ ]:
# Summary spreadsheet (Claude reads this from Drive) + printed grids
!python scripts/summarize_results.py --inputs "$OUT_DIR/main_model_v1_news_r2.json" "$OUT_DIR/main_model_v0_r2.json" "$OUT_DIR/singlescale_v1_news_r2.json" --out "$OUT_DIR/results_summary.xlsx"